# Kaggle Local Test

Simulates the Kaggle Connect-X environment locally using `kaggle-environments==0.1.6`.

Supports two submission formats:
- **`submission/main.py`** (base64 mode, `--base64`) — single file upload, **recommended**
- **`submission/submission.zip`** (zip mode) — kept for reference, does NOT work for Connect-X

The notebook auto-detects which format is present and loads accordingly.

In [ ]:
import sys
import time
import zipfile
import tempfile
import importlib.util
from pathlib import Path

# Project root so src/ imports work
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from kaggle_environments import make, evaluate

print('kaggle-environments loaded')
print('connectx config:', dict(make('connectx').configuration))

## 1. Load agent from submission package

Auto-detects the submission format:
- `submission/main.py` (base64 mode): imported directly — model is embedded, no zip needed.
- `submission/submission.zip` (zip mode): extracted to a temp dir first.

In [ ]:
import os
import importlib.util
import tempfile
import zipfile

SUBMISSION_DIR = PROJECT_ROOT / 'submission'
MAIN_PY  = SUBMISSION_DIR / 'main.py'     # base64 mode (recommended)
ZIP_PATH = SUBMISSION_DIR / 'submission.zip'  # zip mode

if MAIN_PY.exists():
    print(f'Loading from {MAIN_PY} (base64 mode)')
    # Change to a temp dir so the embedded model's temp file lands somewhere clean
    _tmp_dir = tempfile.mkdtemp(prefix='kaggle_test_')
    os.chdir(_tmp_dir)
    spec = importlib.util.spec_from_file_location('kaggle_main', MAIN_PY)

elif ZIP_PATH.exists():
    print(f'Loading from {ZIP_PATH} (zip mode)')
    _tmp_dir = tempfile.mkdtemp(prefix='kaggle_test_')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(_tmp_dir)
    print('Extracted:', [f.name for f in Path(_tmp_dir).iterdir()])
    os.chdir(_tmp_dir)
    spec = importlib.util.spec_from_file_location('kaggle_main', Path(_tmp_dir) / 'main.py')

else:
    raise FileNotFoundError(
        f'No submission found. Run one of:\n'
        f'  python scripts/kaggle_submit.py --model model.onnx --output submission/ --base64\n'
        f'  python scripts/kaggle_submit.py --model model.onnx --output submission/ --zip'
    )

agent_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(agent_module)
my_agent = agent_module.my_agent
print('Agent loaded successfully')

## 2. Smoke test — single move

In [ ]:
from kaggle_environments.utils import structify

env = make('connectx', debug=True)
obs = structify({'board': [0] * 42, 'mark': 1})
conf = structify(dict(env.configuration))

start = time.perf_counter()
col = my_agent(obs, conf)   # first call: loads ONNX session
t1 = time.perf_counter() - start

start = time.perf_counter()
col2 = my_agent(obs, conf)  # second call: session cached
t2 = time.perf_counter() - start

print(f'First call (model load + MCTS): col={col},  {t1:.2f}s')
print(f'Second call (cached):           col={col2}, {t2:.2f}s')
assert col in range(7), f'Invalid column: {col}'

## 3. Full game: agent vs random

In [ ]:
env = make('connectx', debug=True)
env.run([my_agent, 'random'])

final = env.steps[-1]
p1_reward = final[0]['reward']
p2_reward = final[1]['reward']
result = 'WIN' if p1_reward == 1 else ('LOSS' if p1_reward == 0 and p2_reward == 1 else 'DRAW')

print(f'Result (agent as P1): {result}  (rewards: P1={p1_reward}, P2={p2_reward})')
print()
print(env.render(mode='ansi'))

## 4. Win-rate over N games

In [ ]:
N_GAMES = 10  # increase for a more reliable estimate

wins = losses = draws = 0
move_times = []

for game_i in range(N_GAMES):
    env = make('connectx', debug=False)

    # Wrap agent to record per-move timing
    def timed_agent(obs, conf):
        t = time.perf_counter()
        col = my_agent(obs, conf)
        move_times.append(time.perf_counter() - t)
        return col

    env.run([timed_agent, 'random'])
    reward = env.steps[-1][0]['reward']
    if reward == 1:
        wins += 1
    elif reward == 0 and env.steps[-1][1]['reward'] == 1:
        losses += 1
    else:
        draws += 1

    print(f'Game {game_i+1}/{N_GAMES}: {"WIN" if reward==1 else "LOSS" if env.steps[-1][1]["reward"]==1 else "DRAW"}')

print(f'\nResults over {N_GAMES} games (agent as P1 vs random):')
print(f'  Wins:   {wins}/{N_GAMES} ({100*wins/N_GAMES:.0f}%)')
print(f'  Losses: {losses}/{N_GAMES}')
print(f'  Draws:  {draws}/{N_GAMES}')
if move_times:
    import statistics
    print(f'\nPer-move timing (excluding first call):')
    print(f'  median: {statistics.median(move_times[1:]):.3f}s')
    print(f'  max:    {max(move_times[1:]):.3f}s')
    print(f'  Kaggle budget: 2.000s')

## 5. Diagnose: play one move-by-move with board printout

In [ ]:
env = make('connectx', debug=True)

# Step manually to inspect each move
env.reset()
done = False
step = 0

while not done:
    state = env.state[0]
    obs = state['observation']
    conf = structify(dict(env.configuration))
    current_mark = obs['mark']

    if current_mark == 1:
        t = time.perf_counter()
        action = my_agent(structify(obs), conf)
        elapsed = time.perf_counter() - t
        label = f'AlphaZero (P1) → col {action}  [{elapsed:.3f}s]'
    else:
        import random
        legal = [c for c in range(7) if obs['board'][c] == 0]
        action = random.choice(legal)
        label = f'Random (P2) → col {action}'

    env.step([action])
    step += 1
    status = env.state[0]['status']
    done = status in ('DONE', 'ERROR', 'INVALID')

    print(f'--- Step {step}: {label} ---')
    print(env.render(mode='ansi'))

print('Final status:', env.state[0]['status'])
print('Rewards:', [env.state[i]['reward'] for i in range(2)])